In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import plotly.graph_objects as go
import logging
import ast
import math
from collections import Counter
import random
from torch.nn.utils.rnn import pad_sequence

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset

import utils
import z_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(device)

cuda


In [2]:
import numpy as np
import torch
import ast
from torch.utils.data import TensorDataset, DataLoader

def parse_element(element):
    # Only use ast.literal_eval if the element is a string
    if isinstance(element, str):
        return ast.literal_eval(element)
    return element  # Return the element as is if it's already in numerical format

def get_train_loader(batch_size):
    # Load data from files
    spike_num_data = torch.load('../../Data_processed/VAE/sgen_train_spike_num.pt')
    spike_type_data = torch.load('../../Data_processed/VAE/sgen_train_spike_type.pt')
    spike_duration_data = torch.load('../../Data_processed/VAE/sgen_train_spike_duration.pt')
    spike_time_data = torch.load('../../Data_processed/VAE/sgen_train_spike_time.pt')
    spike_magnitude_data = torch.load('../../Data_processed/VAE/sgen_train_spike_magnitude.pt')
    weather_data = torch.load('../../Data_processed/VAE/sgen_train_weather.pt')
    date_data = torch.load('../../Data_processed/VAE/sgen_train_date.pt')
    id_data = torch.load('../../Data_processed/VAE/sgen_train_id.pt')

    # Convert spike_num_data to a tensor
    spike_num_data = torch.FloatTensor(spike_num_data)

    # Convert nested lists into NumPy arrays and then into tensors
    spike_type_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in spike_type_data for inner_list in outer_list]
    spike_type_data_np = np.array(spike_type_data_flat).reshape(len(spike_type_data), len(spike_type_data[0]), -1)
    spike_type_data = torch.FloatTensor(spike_type_data_np)

    spike_duration_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in spike_duration_data for inner_list in outer_list]
    spike_duration_data_np = np.array(spike_duration_data_flat).reshape(len(spike_duration_data), len(spike_duration_data[0]), -1)
    spike_duration_data = torch.FloatTensor(spike_duration_data_np)

    spike_time_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in spike_time_data for inner_list in outer_list]
    spike_time_data_np = np.array(spike_time_data_flat).reshape(len(spike_time_data), len(spike_time_data[0]), -1)
    spike_time_data = torch.FloatTensor(spike_time_data_np)

    spike_magnitude_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in spike_magnitude_data for inner_list in outer_list]
    spike_magnitude_data_np = np.array(spike_magnitude_data_flat).reshape(len(spike_magnitude_data), len(spike_magnitude_data[0]), -1)
    spike_magnitude_data = torch.FloatTensor(spike_magnitude_data_np)

    # Handle weather_data, assuming it might be a mix of strings and arrays
    weather_data_parsed = [[parse_element(inner_list) for inner_list in outer_list] for outer_list in weather_data]
    weather_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in weather_data_parsed for inner_list in outer_list]
    weather_data_np = np.array(weather_data_flat).reshape(len(weather_data), len(weather_data[0]), -1)
    weather_data = torch.FloatTensor(weather_data_np)

    # Handle date_data, assuming it might be a mix of strings and arrays
    date_data_parsed = [[parse_element(inner_list) for inner_list in outer_list] for outer_list in date_data]
    date_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in date_data_parsed for inner_list in outer_list]
    date_data_np = np.array(date_data_flat).reshape(len(date_data), len(date_data[0]), -1)
    date_data = torch.FloatTensor(date_data_np)

    # Handle id_data, assuming it might be a mix of strings and arrays
    id_data_parsed = [[parse_element(inner_list) for inner_list in outer_list] for outer_list in id_data]
    id_data_flat = [np.array(inner_list, dtype=np.float32) for outer_list in id_data_parsed for inner_list in outer_list]
    id_data_np = np.array(id_data_flat).reshape(len(id_data), len(id_data[0]), -1)
    id_data = torch.FloatTensor(id_data_np)

    # Create a TensorDataset and DataLoader
    ds = TensorDataset(spike_num_data, spike_type_data, spike_duration_data, spike_time_data, spike_magnitude_data, weather_data, date_data, id_data)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    return loader

# Example call
train_loader = get_train_loader(128)


In [3]:
num, type, duration, time, magnitude, weather, date, id = next(iter(train_loader))
print('num shape:', num.shape)
print('type shape:', type.shape)
print('duration shape:', duration.shape)
print('time shape:', time.shape)
print('magnitude shape:', magnitude.shape)

print('weather shape:', weather.shape)
print('date shape:', date.shape) 
print('id shape:', id.shape)


num shape: torch.Size([128, 14])
type shape: torch.Size([128, 14, 4])
duration shape: torch.Size([128, 14, 4])
time shape: torch.Size([128, 14, 4])
magnitude shape: torch.Size([128, 14, 4])
weather shape: torch.Size([128, 14, 3])
date shape: torch.Size([128, 14, 2])
id shape: torch.Size([128, 14, 6])


In [6]:
class sp_n_encoder(nn.Module):
    def __init__(self, hidden_size=256, embedding_size=3, lstm_size=32, lstm_layers = 2, output_size=1024):
        super(sp_n_encoder, self).__init__()
        self.n_embedding = nn.Embedding(5, embedding_size)
        self.lstm_size = lstm_size
        self.lstm_layers = lstm_layers
        self.n_lstm = nn.LSTM(embedding_size, lstm_size, num_layers=lstm_layers, batch_first=True)
        self.n_layernorm = nn.LayerNorm(lstm_size)  # Use LayerNorm for LSTM output
        self.n_linear = nn.Sequential(
            nn.Linear(lstm_size, hidden_size),
            nn.SELU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        x = self.n_embedding(x)
        h0 = torch.zeros(self.lstm_layers, x.size(0), self.lstm_size).to(x.device)
        c0 = torch.zeros(self.lstm_layers, x.size(0), self.lstm_size).to(x.device)
        x, _ = self.n_lstm(x, (h0, c0))
        x = self.n_layernorm(x)  # Apply LayerNorm on LSTM output
        x = self.n_linear(x)
        return x


class sp_n_decoder(nn.Module):
    def __init__(self, id_hidden_size = 16, date_hidden_size = 16, weather_lstm_size = 8, weather_hidden_size = 16, fc_hidden_size=8):
        super(sp_n_decoder, self).__init__()
        self.id_net = nn.Sequential(
            nn.Linear(6, id_hidden_size),
            nn.SELU(),
            nn.Linear(id_hidden_size, id_hidden_size),
            nn.SELU(),
            nn.Linear(id_hidden_size, 8)
        )

        self.id_lstm = nn.LSTM(6, hidden_size=16, num_layers=2, batch_first=True)

        self.date_net = nn.Sequential(
            nn.Linear(2, date_hidden_size),
            nn.SELU(),
            nn.Linear(date_hidden_size, date_hidden_size),
            nn.SELU(),
            nn.Linear(date_hidden_size, 256),
        )

        self.weather_lstm = nn.LSTM(3, hidden_size=weather_lstm_size, num_layers=2, batch_first=True)

        self.weather_net = nn.Sequential(
            nn.Linear(3, weather_hidden_size),
            nn.SELU(),
            nn.Linear(weather_hidden_size, weather_hidden_size),
            nn.SELU(),
            nn.Linear(weather_hidden_size, 256),
        )

        self.fc1 = nn.Sequential(
            nn.Linear(512 + 26, fc_hidden_size),
            nn.LayerNorm(fc_hidden_size),
            nn.SELU(),
            nn.Linear(fc_hidden_size, fc_hidden_size),
            nn.LayerNorm(fc_hidden_size),
            nn.SELU(),
            nn.Linear(fc_hidden_size, 64)
        )

        self.fc2 = nn.Sequential(
            nn.Linear(64 + 26, fc_hidden_size),
            nn.LayerNorm(fc_hidden_size),
            nn.SELU(),
            nn.Linear(fc_hidden_size, fc_hidden_size),
            nn.LayerNorm(fc_hidden_size),
            nn.SELU(),
            nn.Linear(fc_hidden_size, 5)# 5
        )

    def forward(self, id, weather, date, z):
        # id_n = self.id_net(id)
        # date_n = self.date_net(date)
        # weather_n = self.weather_net(weather)
        h0_w = torch.zeros(2, weather.size(0), 8).to(weather.device)
        c0_w = torch.zeros(2, weather.size(0), 8).to(weather.device)
        h0_id = torch.zeros(2, id.size(0), 16).to(id.device)
        c0_id = torch.zeros(2, id.size(0), 16).to(id.device)
        w, _ = self.weather_lstm(weather, (h0_w, c0_w))
        id, _ = self.id_lstm(id, (h0_id, c0_id))
        x = torch.cat([id, w, date, z], dim=-1)
        x = self.fc1(x)
        x = torch.cat([id, w, date, x], dim=-1)
        x = self.fc2(x)
        return x

class sp_n_VAE(nn.Module):
    def __init__(self):
        super(sp_n_VAE, self).__init__()
        self.encoder = sp_n_encoder()
        self.decoder = sp_n_decoder()

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) * 2
        return mu + eps * std

    def forward(self, num, type, duration, time, magnitude, weather, date, id):
        out = self.encoder(num)
        mu, logvar = out.chunk(2, dim=2)  # Split output into mu and logvar
        z = self.reparameterize(mu, logvar)

        use_teacher_forcing = True if random.random() < 0.5 else False

        if use_teacher_forcing:
            syn = self.decoder(id, weather, date, z)
        else:
            noise = torch.randn_like(z)
            syn = self.decoder(id, weather, date, noise)
            
        return syn, mu, logvar
    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.orthogonal_(param)  # For input-hidden weights
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param)  # For hidden-hidden weights
                    elif 'bias' in name:
                        nn.init.constant_(param, 1e-3)  # Biases initialized to a small constant
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='selu')
                nn.init.constant_(m.bias, 1e-3)

# Loss function with KL annealing support
def loss_function(recon_x, x, mu, logvar, kl_weight=1.0):
    BCE = nn.functional.cross_entropy(recon_x.view(-1, 5), x.view(-1), reduction='mean')
    # MSE = nn.functional.mse_loss(recon_x, x, reduction='mean')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + kl_weight * KLD
    # return MSE + kl_weight * KLD

In [7]:
model = sp_n_VAE().to(device)
model.init_weights()

optimizer = optim.Adam(model.parameters(), lr=1e-10)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=10)
epochs = 1000

for epoch in range(epochs):
    epoch_loss = 0.0 
    for num, type, duration, time, magnitude, weather, date, id in train_loader:
        
        num = num.to(device).long()
        
        type = type.to(device)
        duration = duration.to(device)
        time = time.to(device)
        magnitude = magnitude.to(device)
        weather = weather.to(device)
        date = date.to(device)
        id = id.to(device)

        optimizer.zero_grad()
        syn, mu, logvar = model(num, type, duration, time, magnitude, weather, date, id)
        kl_weight = min(1.0, (epoch + 1) / 10.0) 

        loss = loss_function(syn, num, mu, logvar, kl_weight)
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    scheduler.step(epoch_loss)
    print('epoch [{}/{}], loss:{:.4f}'.format(epoch+1, epochs, loss.item()))


epoch [1/1000], loss:748.8236


KeyboardInterrupt: 

In [90]:
num, type, duration, time, magnitude, weather, date, id = next(iter(train_loader))
num = num.to(device).long()
type = type.to(device)
duration = duration.to(device)
time = time.to(device)
magnitude = magnitude.to(device)
weather = weather.to(device)
date = date.to(device)
id = id.to(device)

noise = torch.randn(128, 14, 512).to(device)
syn = model.decoder(id, weather, date, noise)
print('syn shape:', syn.shape)
# See the first 5 samples
num = num.cpu().detach().numpy()
for i in range(5):
    print('num:', num[i])
    print('syn:', torch.argmax(syn[i], dim=1))
    print('')

syn shape: torch.Size([128, 14, 5])
num: [1 3 2 2 2 1 2 2 1 2 2 1 4 2]
syn: tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], device='cuda:0')

num: [2 0 2 0 1 2 4 2 2 0 3 2 3 4]
syn: tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], device='cuda:0')

num: [3 4 4 4 4 4 4 3 2 4 4 4 4 3]
syn: tensor([3, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], device='cuda:0')

num: [3 1 3 2 1 2 3 3 1 1 2 2 2 2]
syn: tensor([2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 1], device='cuda:0')

num: [2 0 3 3 3 3 1 3 3 2 3 0 3 1]
syn: tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], device='cuda:0')

